In [1]:
import nltk
import os
import torch
from gensim import downloader
import numpy as np
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader,Dataset
from nltk.tokenize import sent_tokenize
from transformers import BertTokenizer
from nlp import ALICE_URL, WIZARD_URL, download_text
from datasets import load_dataset,Split
from nlp import sentence_tokenize
tokenizer=BertTokenizer('bert.txt')
glove = downloader.load('glove-wiki-gigaword-50')

In [2]:
localfolder = 'texts'
download_text(ALICE_URL, localfolder)
download_text(WIZARD_URL, localfolder)

In [3]:
new_fnames=sentence_tokenize(localfolder)

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/giradasaiteja/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [4]:
dataset=load_dataset(path='csv',data_files=new_fnames,quotechar='\\',split=Split.TRAIN)
dataset[807]

Generating train split: 0 examples [00:00, ? examples/s]

{'sentence': 'he said, turning to Alice:  he had taken his watch out of his pocket, and was looking at it uneasily, shaking it every now and then, and holding it to his ear.',
 'source': 'alice28-1476.txt'}

In [5]:
def isalicelabel(row):
    val=int(row['source']=='alice28-1476.txt')
    return {'labels' : val}
dataset=dataset.map(isalicelabel)

Map:   0%|          | 0/3982 [00:00<?, ? examples/s]

In [6]:
shuffled = dataset.shuffle(seed=42)
dataset = shuffled.train_test_split(test_size=0.2)

In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'source', 'labels'],
        num_rows: 3185
    })
    test: Dataset({
        features: ['sentence', 'source', 'labels'],
        num_rows: 797
    })
})

In [8]:
train_data,test_data=dataset['train'],dataset['test']
train_sentence,test_sentence=train_data['sentence'],test_data['sentence']
train_labels,test_labels=train_data['labels'],test_data['labels']
len(train_sentence),len(test_sentence)

(3185, 797)

In [9]:
train_ids=tokenizer(train_sentence,truncation=True,padding=True,max_length=60,add_special_tokens=False,return_tensors='pt')['input_ids']
test_ids=tokenizer(test_sentence,truncation=True,padding=True,max_length=60,add_special_tokens=False,return_tensors='pt')['input_ids']
train_labels=torch.as_tensor(train_labels).float().view(-1,1)
test_labels=torch.as_tensor(test_labels).float().view(-1,1)
len(train_ids),len(train_labels)

(3185, 3185)

In [10]:
train_tensor_data=TensorDataset(train_ids,train_labels)
test_tensor_data=TensorDataset(test_ids,test_labels)
generator=torch.Generator()
train_loader=DataLoader(train_tensor_data,batch_size=32,shuffle=True,generator=generator)
test_loader=DataLoader(test_tensor_data,batch_size=32)

In [11]:
embeddings=glove.vectors
embeddings = torch.as_tensor(embeddings).float()
torch_embeddings = nn.Embedding.from_pretrained(embeddings)
embeddings

tensor([[ 0.4180,  0.2497, -0.4124,  ..., -0.1841, -0.1151, -0.7858],
        [ 0.0134,  0.2368, -0.1690,  ..., -0.5666,  0.0447,  0.3039],
        [ 0.1516,  0.3018, -0.1676,  ..., -0.3565,  0.0164,  0.1022],
        ...,
        [-0.5118,  0.0587,  1.0913,  ..., -0.2500, -1.1250,  1.5863],
        [-0.7590, -0.4743,  0.4737,  ...,  0.7895, -0.0141,  0.6448],
        [ 0.0726, -0.5139,  0.4728,  ..., -0.1891, -0.5902,  0.5556]])

In [12]:
boe_mean=nn.EmbeddingBag.from_pretrained(embeddings,mode="mean")

In [35]:
model=nn.Sequential(
    boe_mean,
    nn.Linear(boe_mean.embedding_dim,128),
    nn.ReLU(),
    nn.Linear(128,1))
criterion=nn.BCEWithLogitsLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.009)

In [27]:
class GloveClassifier(nn.Module):
    def __init__(self, embedding_layer, embed_dim, num_classes):
        super().__init__()
        self.embedding = embedding_layer
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, input_ids):
        x = self.embedding(input_ids)      # (B, L, D)
        x = x.mean(dim=1)                  # (B, D)
        out = self.fc(x)                   # (B, C)
        return out

In [36]:
%%time
#criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs=98
for epoch in range(epochs):
    for X, y in train_loader:
        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
    print(loss)

tensor(0.7004, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.4682, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.4843, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.4265, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.3839, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.6451, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.4908, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.2803, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.3042, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.5926, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.6009, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.3455, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.1666, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.3853, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.4602, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.3754, grad_fn=<BinaryCrossEntro

In [37]:
model.eval()
with torch.no_grad():
    preds = model(test_ids)
    accuracy = (preds.argmax(1) == test_labels).float().mean()
accuracy

tensor(0.6136)